# 04 · Lectura de archivos tabulados y operaciones básicas

Trabajaremos con tres archivos: sismos, meteorología de Medellín y pingüinos. Ejecuta las celdas en orden. Usaremos **R base**, sin instalar paquetes adicionales.

## Objetivos y recorrido

- Leer un CSV reconociendo separador, encabezado, decimales y ausentes.
- Revisar qué representa cada fila, los tipos de columnas y sus unidades.
- Seleccionar columnas, filtrar filas y ordenar una tabla.
- Construir fechas a partir del año y el día del año.
- Calcular conteos y resúmenes sencillos por grupos.

**Recorrido:** archivo → tabla → inspección → selección y limpieza → operaciones → resumen.


## ¿Qué es un archivo tabulado?

Una tabla organiza observaciones en filas y variables en columnas. Un CSV guarda esa tabla como texto, normalmente separando sus campos con comas. La extensión no garantiza que todos los archivos usen el mismo separador o tengan la misma cabecera.

Antes de leer, revisa:

| Aspecto | Qué debemos reconocer |
|---|---|
| Encabezado | La línea que contiene los nombres de las columnas. |
| Separador | Coma, punto y coma o tabulación. |
| Decimal | Punto o coma dentro de los números. |
| Ausentes | Campos vacíos, `NA` o un código documentado como `-999`. |
| Metadatos | Líneas previas que describen fuente, lugar, unidades o fechas. |

Una fila de sismos es un evento; una fila meteorológica es un día en una ubicación; una fila de pingüinos es un individuo muestreado.


## Carpeta de trabajo y rutas

`getwd()` muestra desde dónde busca R los archivos. La ruta `../../data/raw` parte de `estudiantes/clases/`: sube dos carpetas y entra en `data/raw/`.

- **En local:** si trabajas desde `estudiantes/clases/`, conserva `carpeta_datos <- "../../data/raw"`. Si trabajas desde la raíz del repositorio, cámbiala a `"data/raw"`.
- **En Colab:** selecciona R y carga los tres CSV en el panel de archivos. Cambia `carpeta_datos` a `"/content"`.

Conserva los nombres originales, incluidos espacios y tildes. `file.path()` combina una carpeta y un nombre.


In [ ]:
getwd()


In [ ]:
carpeta_datos <- "../../data/raw"
archivo_sismos <- file.path(carpeta_datos, "Earthquakes in Colombia (from 2000).csv")
archivo_meteo <- file.path(carpeta_datos, "Datos hidrometeorológicos Medellín.csv")
archivo_pinguinos <- file.path(carpeta_datos, "DB Pinguinos.csv")
file.exists(c(archivo_sismos, archivo_meteo, archivo_pinguinos))


Los tres valores deben ser `TRUE`. Si alguno es `FALSE`, revisa la carpeta y el nombre antes de continuar. Para mirar texto sin convertirlo aún en tabla usamos `readLines()`; el argumento `n` limita cuántas líneas se muestran.


In [ ]:
readLines(archivo_sismos, n = 3, encoding = "UTF-8")


## Estructura básica de lectura

```r
tabla <- read.csv(
  "ruta/al/archivo.csv",
  header = TRUE,
  sep = ",",
  dec = ".",
  na.strings = c("", "NA"),
  fileEncoding = "UTF-8"
)
```

`read.csv()` devuelve un `data.frame`. `header`, `sep` y `dec` ya tienen esos valores por defecto, pero escribirlos permite entender el formato. `na.strings` indica qué textos se interpretarán como ausentes. Usaremos también `stringsAsFactors = FALSE` para conservar las etiquetas como texto.

Para otros formatos: `read.csv2()` usa por defecto `sep = ";"` y `dec = ","`; `read.delim()` usa tabulaciones. Revisa el archivo antes de escoger la función.


## 1. Sismos: profundidad y magnitud

**Fuente: USGS — Catálogo de terremotos.** Registra fecha, coordenadas, profundidad y magnitud; permite descargar CSV, GeoJSON y QuakeML. [Buscar y descargar terremotos](https://earthquake.usgs.gov/earthquakes/search/) · [API de datos](https://earthquake.usgs.gov/fdsnws/event/1/1).

**Pregunta:** ¿qué registros tienen Colombia en su localización y cuáles son sus profundidades y magnitudes?

`depth` es profundidad en km y `mag` es magnitud (sin unidad física). `magType` identifica la escala reportada: no todos los registros usan necesariamente el mismo tipo de magnitud.


In [ ]:
sismos <- read.csv(
  archivo_sismos, header = TRUE, sep = ",", dec = ".",
  na.strings = c("", "NA"), fileEncoding = "UTF-8",
  stringsAsFactors = FALSE
)
head(sismos)


### Inspeccionar antes de calcular

`head()` muestra las primeras filas; `dim()` devuelve filas y columnas; `names()` muestra nombres; `str()` describe tipos. `summary()` resume números y ayuda a detectar rangos inesperados. Usa las mismas consultas en cualquier tabla que leas.


In [ ]:
dim(sismos)


In [ ]:
names(sismos)


In [ ]:
str(sismos)


### Seleccionar columnas y limpiar la tabla

```r
tabla[, c("columna_1", "columna_2")]  # Columnas elegidas, todas las filas
tabla[condicion, ]                      # Filas elegidas, todas las columnas
tabla[order(tabla$variable), ]          # Filas ordenadas de menor a mayor
```

Aunque el nombre del archivo menciona Colombia, también contiene localizaciones de países vecinos. Primero conservaremos las columnas de interés y después filtraremos la columna `place`, que contiene la descripción de la ubicación.


In [ ]:
sismos_filtro_columnas <- sismos[, c("time", "depth", "mag", "magType", "place")]
head(sismos_filtro_columnas)


### Buscar “Colombia” en la localización

`grepl("Colombia", texto, fixed = TRUE)` devuelve `TRUE` cuando encuentra ese texto y `FALSE` cuando no lo encuentra. `fixed = TRUE` indica que buscamos el texto literal. Excluimos explícitamente las localizaciones ausentes.

Guardamos las filas seleccionadas en una nueva tabla y mantenemos `sismos` como copia de la lectura original. Este filtro usa la descripción del catálogo; no equivale a comprobar un límite geográfico oficial.


In [ ]:
filtro_colombia <- !is.na(sismos_filtro_columnas$place) & grepl("Colombia", sismos_filtro_columnas$place, fixed = TRUE)
sismos_colombia <- sismos_filtro_columnas[filtro_colombia, ]
print(c(
  registros_originales = nrow(sismos_filtro_columnas),
  registros_colombia = nrow(sismos_colombia),
  registros_excluidos = nrow(sismos_filtro_columnas) - nrow(sismos_colombia)
))

In [ ]:
head(sismos_colombia)


In [ ]:
table(sismos_colombia$magType, useNA = "ifany")


In [ ]:
print(c(
  profundidad_ausente = sum(is.na(sismos_colombia$depth)),
  magnitud_ausente = sum(is.na(sismos_colombia$mag))
))


### Una función de resumen reutilizable

Retomamos la clase 03. Esta función recibe un vector numérico y devuelve un vector con nombres: cuántos valores se conocen, cuántos faltan y varias medidas descriptivas. Si no hay observaciones, entrega ausentes para las medidas.

La **media** es el promedio; la **mediana** divide los datos ordenados en dos mitades; la **desviación estándar** describe dispersión alrededor de la media; Q1 y Q3 delimitan el 50 % central. Las medidas conservan la unidad de la variable. La desviación estándar requiere al menos dos valores conocidos.


In [ ]:
resumir_variable <- function(x) {
  n_validos <- sum(!is.na(x))
  if (n_validos == 0) {
    return(c(n_validos = 0, n_ausentes = length(x),
             minimo = NA_real_, q1 = NA_real_, mediana = NA_real_,
             media = NA_real_, q3 = NA_real_, maximo = NA_real_, sd = NA_real_))
  }
  return(c(
    n_validos = n_validos,
    n_ausentes = sum(is.na(x)),
    minimo = min(x, na.rm = TRUE),
    q1 = unname(quantile(x, 0.25, na.rm = TRUE)),
    mediana = median(x, na.rm = TRUE),
    media = mean(x, na.rm = TRUE),
    q3 = unname(quantile(x, 0.75, na.rm = TRUE)),
    maximo = max(x, na.rm = TRUE),
    sd = sd(x, na.rm = TRUE)
  ))
}


In [ ]:
print(quantile(sismos_colombia$mag, 0.25, na.rm = TRUE))

In [ ]:
resumen_sismos <- rbind(
  profundidad_km = resumir_variable(sismos_colombia$depth),
  magnitud = resumir_variable(sismos_colombia$mag)
)
round(resumen_sismos, 2)


In [ ]:
summary(sismos_colombia[, c("depth", "mag")])

### Filtrar por magnitud y ordenar resultados

Seleccionaremos eventos con magnitud conocida mayor o igual a 5. La condición combina `!is.na()` con la comparación usando `&`. El umbral es un criterio de selección para este ejercicio.

Después ordenaremos las filas por magnitud descendente y consultaremos las cinco primeras con `head()`. Profundidad y magnitud son medidas distintas: compara sus resúmenes por separado.


In [ ]:
sismos_mayores <- sismos_colombia[
  !is.na(sismos_colombia$mag) & sismos_colombia$mag >= 5,
]
nrow(sismos_mayores)


In [ ]:
sismos_ordenados <- sismos_mayores[order(sismos_mayores$mag, decreasing = TRUE), ]
head(sismos_ordenados, 5)


## 2. Meteorología: precipitación y temperatura

**Fuente: NASA POWER — Meteorología e hidrología.** Ofrece series diarias por coordenada en CSV, JSON y NetCDF. [Consultar API y ejemplos](https://power.larc.nasa.gov/docs/services/api/temporal/daily/) · [Visor para consultar y descargar](https://power.larc.nasa.gov/data-access-viewer/).

**Pregunta:** ¿cómo cambia la temperatura media de un mismo mes entre los años disponibles?

El encabezado del archivo describe la ubicación (6.2446, −75.5451), el periodo 2000-01-01 a 2026-09-30 y la resolución de la fuente MERRA-2. Son datos de una celda del producto, no lecturas directas de una estación puntual de Medellín.


In [ ]:
readLines(archivo_meteo, n = 15, encoding = "UTF-8")


### Saltar metadatos y reconocer el código de ausentes

En **este archivo** hay 12 líneas previas y la línea 13 contiene los nombres de columnas. `skip = 12` omite solo esas líneas; `header = TRUE` usa la siguiente como encabezado. El valor `-999` significa dato ausente según los metadatos: lo declaramos al leer para que no altere los promedios.

Si descargas otro archivo, vuelve a mirar sus primeras líneas antes de reutilizar `skip`.

| Columna | Significado |
|---|---|
| `YEAR` | Año. |
| `DOY` | Día del año: 1 corresponde al 1 de enero. |
| `T2M` | Temperatura media diaria a 2 m (°C). |
| `PRECTOTCORR` | Precipitación corregida (mm/día). |
| `WS2M`, `GWETTOP` | Viento a 2 m y humedad superficial del suelo; quedan fuera de esta exploración. |

Este CSV contiene `T2M`, no temperaturas máximas y mínimas.


In [ ]:
meteo <- read.csv(
  archivo_meteo, skip = 12, header = TRUE, sep = ",", dec = ".",
  na.strings = c("", "NA", "-999"), fileEncoding = "UTF-8",
  stringsAsFactors = FALSE
)
head(meteo)


In [ ]:
str(meteo)


In [ ]:
dim(meteo)


### Seleccionar y dar nombres claros

Trabajaremos con año, día del año y las dos variables de interés. Renombrar una copia permite expresar las unidades sin modificar el archivo original.


In [ ]:
meteo_eda <- meteo[, c("YEAR", "DOY", "T2M", "PRECTOTCORR")]
names(meteo_eda) <- c("anio", "dia_anio", "temperatura_c", "precipitacion_mm_dia")
head(meteo_eda)


In [ ]:
resumen_meteo <- rbind(
  temperatura_c = resumir_variable(meteo_eda$temperatura_c),
  precipitacion_mm_dia = resumir_variable(meteo_eda$precipitacion_mm_dia)
)
round(resumen_meteo, 2)


### Un filtro sencillo: días con precipitación positiva

El cero es un registro conocido sin precipitación, no un ausente. Excluimos `NA` explícitamente al construir el filtro; así evitamos seleccionar filas desconocidas por una comparación con un ausente.

Calcularemos la proporción de días con precipitación positiva **entre los días con precipitación conocida**. Es una descripción del producto y del periodo, no una definición universal de “día lluvioso”.


In [ ]:
dias_con_precipitacion <- meteo_eda[
  !is.na(meteo_eda$precipitacion_mm_dia) & meteo_eda$precipitacion_mm_dia > 0,
]
n_dias_conocidos <- sum(!is.na(meteo_eda$precipitacion_mm_dia))
c(dias_conocidos = n_dias_conocidos,
  dias_con_precipitacion = nrow(dias_con_precipitacion),
  porcentaje = 100 * nrow(dias_con_precipitacion) / n_dias_conocidos)


### Construir una fecha a partir del año y DOY

`DOY` es el día del año: 1 representa el 1 de enero. `as.Date()` convierte texto en fecha. Construimos el 1 de enero de cada año y sumamos `dia_anio - 1` días: el día 1 no debe desplazar esa fecha. R maneja los años bisiestos en esta suma.

Guardamos la fecha en una columna nueva y ordenamos la tabla cronológicamente.


In [ ]:
meteo_eda$fecha <- as.Date(paste0(meteo_eda$anio, "-01-01")) + meteo_eda$dia_anio - 1
meteo_eda <- meteo_eda[order(meteo_eda$fecha), ]
head(meteo_eda)
range(meteo_eda$fecha)


### Identificar el mes y seleccionar el mismo mes en todos los años

`format(fecha, "%m")` extrae el mes como texto (`"01"`, ..., `"12"`). `as.integer()` lo convierte en un número entre 1 y 12 para facilitar el filtro.

Elegiremos **enero** (`mes_elegido <- 1`), disponible en todos los años del archivo. Puedes cambiarlo para explorar otro mes. El archivo termina en septiembre de 2026: octubre, noviembre y diciembre no incluyen ese último año.


In [ ]:
meteo_eda$mes <- as.integer(format(meteo_eda$fecha, "%m"))
mes_elegido <- 1
datos_mes <- meteo_eda[meteo_eda$mes == mes_elegido, ]
head(datos_mes)
table(datos_mes$anio)


### Comparar la temperatura de ese mes entre años

```r
aggregate(variable ~ grupo, data = tabla, FUN = mean)
```

`aggregate()` separa la tabla por el grupo indicado a la derecha de `~` y aplica la función a la variable de la izquierda. Aquí calculamos la temperatura media de enero para cada año.

Primero excluimos solo los ausentes de temperatura para saber cuántos días aportan al promedio. Una media basada en menos días puede no ser comparable con la de un mes completo.


In [ ]:
mes_temperatura <- datos_mes[!is.na(datos_mes$temperatura_c), ]
table(mes_temperatura$anio)


In [ ]:
temperatura_por_anio <- aggregate(
  temperatura_c ~ anio, data = mes_temperatura, FUN = mean
)
temperatura_por_anio


También podemos usar `FUN = sum` para sumar la precipitación diaria del mes. Al sumar los valores diarios, el resultado se expresa en **mm del periodo**, no en mm/día. Consultamos cuántos días tienen precipitación conocida: con días ausentes, la suma sería parcial.


In [ ]:
mes_precipitacion <- datos_mes[!is.na(datos_mes$precipitacion_mm_dia), ]
table(mes_precipitacion$anio)


In [ ]:
precipitacion_por_anio <- aggregate(
  precipitacion_mm_dia ~ anio, data = mes_precipitacion, FUN = sum
)
names(precipitacion_por_anio)[2] <- "precipitacion_mes_mm"
precipitacion_por_anio


### Consultar los años con mayor temperatura media del mes

Ordenamos el resumen, no los registros diarios. Cada fila de este resultado representa ahora un año. La comparación describe variación entre años; por sí sola no demuestra una tendencia climática.


In [ ]:
temperatura_ordenada <- temperatura_por_anio[
  order(temperatura_por_anio$temperatura_c, decreasing = TRUE),
]
head(temperatura_ordenada, 5)


## 3. Pingüinos: explorar todas las columnas

**Fuente: Palmer Penguins — Morfometría de pingüinos.** Incluye especie, isla, sexo y medidas corporales, con algunos valores ausentes; está disponible como CSV y como paquete de R. [Dataset y documentación](https://github.com/allisonhorst/palmerpenguins).

**Pregunta:** ¿qué especies y grupos aparecen? ¿Cómo varían las medidas corporales entre especies?

Exploraremos todas las columnas, pero cada una exige una operación adecuada. No tiene sentido calcular una media de nombres de especies, islas o sexo, ni tratar el año como una medida corporal.


In [ ]:
pinguinos <- read.csv(
  archivo_pinguinos, header = TRUE, sep = ",", dec = ".",
  na.strings = c("", "NA"), fileEncoding = "UTF-8",
  stringsAsFactors = FALSE
)
head(pinguinos)


In [ ]:
str(pinguinos)


In [ ]:
dim(pinguinos)


| Columna | Tipo de variable y uso |
|---|---|
| `species` | Especie: categoría; contar y comparar grupos. |
| `island` | Isla: categoría; revisar composición del muestreo. |
| `sex` | Sexo registrado: categoría con ausentes. |
| `bill_length_mm` | Longitud del pico (mm): resumir distribución. |
| `bill_depth_mm` | Profundidad del pico (mm): resumir distribución. |
| `flipper_length_mm` | Longitud de la aleta (mm): resumir distribución. |
| `body_mass_g` | Masa corporal (g): resumir distribución. |
| `year` | Año de muestreo: revisar cuántos registros hay por año. |

`table()` cuenta categorías; `useNA = "ifany"` muestra también los ausentes cuando existan. Un conteo del archivo describe la muestra, no la abundancia de una población.


In [ ]:
table(pinguinos$species, useNA = "ifany")


In [ ]:
table(pinguinos$island, useNA = "ifany")


In [ ]:
table(pinguinos$sex, useNA = "ifany")


In [ ]:
table(pinguinos$year, useNA = "ifany")


### Cruce de categorías

`table(variable_1, variable_2)` cuenta las combinaciones. ¿Están todas las especies representadas en todas las islas? Esto ayuda a reconocer que las diferencias entre grupos pueden coincidir con diferencias en los lugares de muestreo.


In [ ]:
table(especie = pinguinos$species, isla = pinguinos$island)


### Ausentes por columna y resumen de las cuatro medidas

Usaremos un `for` para revisar cada columna, siguiendo la clase de funciones. `pinguinos[[nombre]]` extrae la columna cuyo nombre está guardado en una variable. Después reunimos las cuatro salidas de `resumir_variable()` con `rbind()`.


In [ ]:
ausentes <- integer(ncol(pinguinos))
names(ausentes) <- names(pinguinos)
for (nombre in names(pinguinos)) {
  ausentes[nombre] <- sum(is.na(pinguinos[[nombre]]))
}
ausentes


In [ ]:
resumen_medidas <- rbind(
  longitud_pico_mm = resumir_variable(pinguinos$bill_length_mm),
  profundidad_pico_mm = resumir_variable(pinguinos$bill_depth_mm),
  longitud_aleta_mm = resumir_variable(pinguinos$flipper_length_mm),
  masa_g = resumir_variable(pinguinos$body_mass_g)
)
round(resumen_medidas, 2)


### Resumir una medida por especie

`aggregate()` separa los datos por grupo y aplica una función a cada grupo. Comenzamos con la masa conocida y la especie conocida para hacer explícito qué registros participan. El número de filas de cada grupo debe acompañar la comparación de promedios.

```r
aggregate(medida ~ grupo, data = tabla, FUN = mean)
```


In [ ]:
pinguinos_masa <- pinguinos[complete.cases(pinguinos[, c("species", "body_mass_g")]), ]
table(pinguinos_masa$species)


In [ ]:
aggregate(body_mass_g ~ species, data = pinguinos_masa, FUN = mean)


### Seleccionar una especie y conservar columnas de interés

Filtramos `"Adelie"` y seleccionamos isla, sexo y masa. Combinamos selección de filas y columnas en `tabla[filas, columnas]`. La tabla conserva los ausentes: el filtro de especie no implica eliminar mediciones faltantes.


In [ ]:
adelie <- pinguinos[
  !is.na(pinguinos$species) & pinguinos$species == "Adelie",
  c("island", "sex", "body_mass_g")
]
head(adelie)
nrow(adelie)


## Taller de la semana

Continúa con el [Taller 3](../talleres/taller_03.ipynb) · [Abrir en Colab](https://colab.research.google.com/github/eamontoyaa/EAFIT_herramientas_computacionales_modulo_R/blob/main/estudiantes/talleres/taller_03.ipynb).
